In [ ]:
# ══════════════════════════════════════════════
# [Cell 1] 환경 설정 및 패키지 설치
# ══════════════════════════════════════════════

# GPU 확인
!nvidia-smi

# 필수 패키지 설치
!pip install -q transformers>=4.45.0
!pip install -q accelerate>=0.34.0
!pip install -q peft>=0.13.0
!pip install -q bitsandbytes>=0.44.0
!pip install -q qwen-vl-utils
!pip install -q flash-attn --no-build-isolation

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ 설치 완료!")


In [ ]:
# ══════════════════════════════════════════════
# [Cell 2] 데이터 경로 설정 및 검증
# ══════════════════════════════════════════════

import os
from pathlib import Path

# ★ 본인 드라이브 경로로 수정
BASE_DIR = Path("/content/drive/MyDrive/ssafy_vqa")

TRAIN_CSV = BASE_DIR / "train_final.csv"
TEST_CSV = BASE_DIR / "test.csv"

# 검증
assert TRAIN_CSV.exists(), f"❌ {TRAIN_CSV} 없음"
assert TEST_CSV.exists(), f"❌ {TEST_CSV} 없음"

import pandas as pd
df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"✅ Train: {len(df):,}건")
print(f"✅ Test: {len(test_df):,}건")
print(f"\n📊 Answer 분포:")
print(df["answer"].value_counts().sort_index())


In [ ]:
"""
==========================================================================
  [Cell 3] Qwen2-VL-7B H100 최고 성능 학습
==========================================================================

  H100 80GB 환경 최적화:
  - 7B 모델 풀 BF16 (양자화 없음)
  - Flash Attention 2
  - 배치 사이즈 4 + GradAccum 8 = 실효 배치 32
  - 고해상도 이미지 (최대 2048 patches)
  - LoRA r=64, alpha=128 (대용량)
  - 5 Epoch 학습

==========================================================================
"""

import os
import gc
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════
# 경로 설정
# ══════════════════════════════════════════════
BASE_DIR = Path("/content/drive/MyDrive/ssafy_vqa")
TRAIN_CSV = BASE_DIR / "train_final.csv"
TEST_CSV = BASE_DIR / "test.csv"
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

OUTPUT_DIR = BASE_DIR / "vqa_h100_7b"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SUBMISSION_DIR = OUTPUT_DIR / "submissions"
ADAPTER_DIR = OUTPUT_DIR / "adapters"

for d in [OUTPUT_DIR, CHECKPOINT_DIR, SUBMISSION_DIR, ADAPTER_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ══════════════════════════════════════════════
# [H100 최고 성능] 하이퍼파라미터
# ══════════════════════════════════════════════
NUM_EPOCHS = 5
BATCH_SIZE = 4                  # H100은 배치 4 가능
GRAD_ACCUM = 8                  # 실효 배치 = 32
LEARNING_RATE = 2e-5            # 7B 최적 LR
WEIGHT_DECAY = 0.01
SEED = 42
LABEL_SMOOTHING = 0.1

# [H100] 대용량 LoRA
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.1

# [H100] 고해상도 이미지
MIN_PIXELS = 256 * 28 * 28      # 200,704
MAX_PIXELS = 2048 * 28 * 28     # 1,605,632


# ══════════════════════════════════════════════
# 유틸리티
# ══════════════════════════════════════════════
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()


def build_prompt(row):
    """모호한 정답 대응 강화 프롬프트"""
    return (
        "You are a recycling classification expert helping with a survey-based challenge.\n"
        "The correct answer is determined by what most people would choose.\n"
        "Look at the image carefully and select the most reasonable answer.\n"
        "If none of the options seem perfect, choose the MOST SIMILAR or CLOSEST one.\n"
        "Think about what an average person would likely select.\n\n"
        f"Question: {row['question']}\n\n"
        f"Options:\n"
        f"a) {row['a']}\n"
        f"b) {row['b']}\n"
        f"c) {row['c']}\n"
        f"d) {row['d']}\n\n"
        f"Based on common sense and what most people would choose, "
        f"answer with only one letter (a, b, c, or d):"
    )


def get_answer_token_ids(processor):
    tokenizer = processor.tokenizer
    ids = {}
    for ch in ["a", "b", "c", "d"]:
        token_ids = tokenizer.encode(ch, add_special_tokens=False)
        ids[ch] = token_ids[-1]
    return ids


# ══════════════════════════════════════════════
# 데이터셋
# ══════════════════════════════════════════════
class VQATrainDataset(Dataset):
    def __init__(self, df, base_dir):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = Path(row["path"])
        if not img_path.is_absolute():
            img_path = self.base_dir / img_path
        
        prompt = build_prompt(row)
        answer = str(row["answer"]).strip().lower()

        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": str(img_path)},
                {"type": "text", "text": prompt},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": answer},
            ]},
        ]
        
        return {
            "messages": messages,
            "image_path": str(img_path),
            "answer": answer,
        }


class VQATestDataset(Dataset):
    def __init__(self, df, base_dir):
        self.df = df.reset_index(drop=True)
        self.base_dir = base_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = Path(row["path"])
        if not img_path.is_absolute():
            img_path = self.base_dir / img_path
        
        prompt = build_prompt(row)

        messages = [
            {"role": "user", "content": [
                {"type": "image", "image": str(img_path)},
                {"type": "text", "text": prompt},
            ]},
        ]
        
        return {
            "id": row["id"],
            "messages": messages,
            "image_path": str(img_path),
        }


# ══════════════════════════════════════════════
# DataCollator + Label Smoothing
# ══════════════════════════════════════════════
class VQADataCollator:
    def __init__(self, processor, label_smoothing=0.1):
        self.processor = processor
        self.label_smoothing = label_smoothing
        
        self.answer_token_ids = {}
        for ch in ["a", "b", "c", "d"]:
            tids = processor.tokenizer.encode(ch, add_special_tokens=False)
            self.answer_token_ids[ch] = tids[-1]

    def __call__(self, features):
        if isinstance(features, dict):
            features = [features]

        texts_full = []
        texts_prompt = []
        images = []
        answers = []

        for f in features:
            msgs = f["messages"]
            
            full = self.processor.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
            texts_full.append(full)

            user_only = [m for m in msgs if m["role"] != "assistant"]
            prompt = self.processor.apply_chat_template(
                user_only, tokenize=False, add_generation_prompt=True
            )
            texts_prompt.append(prompt)

            img = Image.open(f["image_path"]).convert("RGB")
            images.append(img)
            answers.append(f["answer"])

        inputs = self.processor(
            text=texts_full,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        prompt_inputs = self.processor(
            text=texts_prompt,
            images=images,
            padding=False,
            return_tensors=None,
        )

        labels = inputs["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id

        if pad_id is not None:
            labels[labels == pad_id] = -100

        bs, seq_len = labels.shape
        smooth_info = []

        for i, pids in enumerate(prompt_inputs["input_ids"]):
            plen = len(pids)
            
            if pad_id is not None:
                mask = inputs["input_ids"][i] != pad_id
                nonpad = mask.nonzero(as_tuple=True)[0]
                start = nonpad[0].item() if len(nonpad) > 0 else 0
            else:
                start = 0
            
            labels[i, start:start+plen] = -100
            
            ans_pos = start + plen
            if ans_pos < seq_len:
                smooth_info.append({
                    "batch_idx": i,
                    "position": ans_pos,
                    "correct": answers[i],
                })

        inputs["labels"] = labels
        inputs["smooth_info"] = smooth_info
        inputs["answer_ids"] = self.answer_token_ids
        inputs["label_smoothing"] = self.label_smoothing
        
        return inputs


# ══════════════════════════════════════════════
# Custom Trainer
# ══════════════════════════════════════════════
class VQATrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        smooth_info = inputs.pop("smooth_info", [])
        answer_ids = inputs.pop("answer_ids", {})
        label_smoothing = inputs.pop("label_smoothing", 0.1)
        
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]
        
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        
        bs, seq_len, vocab = shift_logits.shape
        flat_logits = shift_logits.view(-1, vocab)
        flat_labels = shift_labels.view(-1)
        
        base_loss = F.cross_entropy(flat_logits, flat_labels, ignore_index=-100)
        
        if smooth_info and label_smoothing > 0:
            smooth_loss = torch.tensor(0.0, device=logits.device, dtype=logits.dtype)
            
            for info in smooth_info:
                i = info["batch_idx"]
                pos = info["position"] - 1
                correct = info["correct"]
                
                if pos < 0 or pos >= seq_len:
                    continue
                
                pos_logits = shift_logits[i, pos]
                soft_target = torch.zeros(vocab, device=logits.device, dtype=logits.dtype)
                
                for ch, tid in answer_ids.items():
                    if ch == correct:
                        soft_target[tid] = 1.0 - label_smoothing
                    else:
                        soft_target[tid] = label_smoothing / 3.0
                
                log_probs = F.log_softmax(pos_logits, dim=-1)
                smooth_loss += F.kl_div(log_probs, soft_target, reduction='sum')
            
            if len(smooth_info) > 0:
                smooth_loss = smooth_loss / len(smooth_info)
            
            loss = 0.5 * base_loss + 0.5 * smooth_loss
        else:
            loss = base_loss
        
        return (loss, outputs) if return_outputs else loss


# ══════════════════════════════════════════════
# 추론 (배치 처리)
# ══════════════════════════════════════════════
def run_inference(model, processor, test_df, batch_size=8):
    """H100용 배치 추론"""
    model.eval()
    
    answer_ids = get_answer_token_ids(processor)
    tid_list = [answer_ids[c] for c in "abcd"]
    
    dataset = VQATestDataset(test_df, BASE_DIR)
    results = []
    
    for start in tqdm(range(0, len(dataset), batch_size), desc="추론"):
        end = min(start + batch_size, len(dataset))
        batch_items = [dataset[i] for i in range(start, end)]
        
        texts = []
        images = []
        ids = []
        
        for item in batch_items:
            txt = processor.apply_chat_template(
                item["messages"], tokenize=False, add_generation_prompt=True
            )
            texts.append(txt)
            images.append(Image.open(item["image_path"]).convert("RGB"))
            ids.append(item["id"])
        
        inputs = processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        for i, sample_id in enumerate(ids):
            attn = inputs["attention_mask"][i]
            last_pos = attn.sum().item() - 1
            logits_at_last = outputs.logits[i, last_pos]
            
            scores = [logits_at_last[t].item() for t in tid_list]
            answer = "abcd"[scores.index(max(scores))]
            
            results.append({"id": sample_id, "answer": answer})
        
        del inputs, outputs
        
        if start % 500 == 0:
            cleanup()
    
    return pd.DataFrame(results)


# ══════════════════════════════════════════════
# 콜백
# ══════════════════════════════════════════════
class EpochCallback(TrainerCallback):
    def __init__(self, processor, test_df):
        self.processor = processor
        self.test_df = test_df
    
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(state.epoch)
        
        print(f"\n{'='*60}")
        print(f"  📌 Epoch {epoch} 완료")
        print(f"{'='*60}")
        
        cleanup()
        
        # 어댑터 저장
        adapter_path = ADAPTER_DIR / f"epoch_{epoch}"
        adapter_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(adapter_path))
        self.processor.save_pretrained(str(adapter_path))
        print(f"  ✅ 어댑터: {adapter_path}")
        
        # 추론
        print(f"  🔄 추론 중... ({len(self.test_df):,}건)")
        result_df = run_inference(model, self.processor, self.test_df, batch_size=8)
        
        # CSV 저장
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        csv_path = SUBMISSION_DIR / f"submission_epoch_{epoch}_{timestamp}.csv"
        result_df.to_csv(csv_path, index=False)
        
        print(f"  ✅ CSV: {csv_path.name}")
        print(f"\n  📊 답변 분포:")
        for ans, cnt in result_df["answer"].value_counts().sort_index().items():
            print(f"     {ans}: {cnt:,}건 ({cnt/len(result_df)*100:.1f}%)")
        
        print(f"\n{'='*60}\n")
        
        cleanup()
        return control


# ══════════════════════════════════════════════
# 메인 학습 함수
# ══════════════════════════════════════════════
def train():
    cleanup()
    torch.manual_seed(SEED)
    
    print("\n" + "=" * 70)
    print("  Qwen2-VL-7B H100 최고 성능 학습")
    print("=" * 70)
    print(f"\n  📁 저장: {OUTPUT_DIR}")
    print(f"  🎲 Seed: {SEED}")
    print(f"  📊 Epochs: {NUM_EPOCHS}")
    print(f"  📦 Batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
    print(f"  📈 LR: {LEARNING_RATE}")
    print(f"  🔧 LoRA: r={LORA_R}, alpha={LORA_ALPHA}")
    print(f"  🖼️ Pixels: {MIN_PIXELS:,} ~ {MAX_PIXELS:,}")
    
    # ══════════════════════════════════════════════
    # 1. 데이터 로드
    # ══════════════════════════════════════════════
    print("\n[1/5] 데이터 로드...")
    df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    
    train_df = df.sample(frac=0.95, random_state=SEED)
    eval_df = df.drop(train_df.index)
    
    train_dataset = VQATrainDataset(train_df, BASE_DIR)
    eval_dataset = VQATrainDataset(eval_df, BASE_DIR)
    
    total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
    warmup_steps = int(total_steps * 0.03)
    
    print(f"  Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,} | Test: {len(test_df):,}")
    print(f"  Steps: {total_steps:,} | Warmup: {warmup_steps:,}")
    
    # ══════════════════════════════════════════════
    # 2. Processor 로드
    # ══════════════════════════════════════════════
    print("\n[2/5] Processor 로드...")
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )
    data_collator = VQADataCollator(processor, label_smoothing=LABEL_SMOOTHING)
    print("  ✅ 완료")
    
    # ══════════════════════════════════════════════
    # 3. 모델 로드 (H100 최적화)
    # ══════════════════════════════════════════════
    print("\n[3/5] 모델 로드 (7B + Flash Attention 2)...")
    
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",  # H100 최적화
    )
    
    model.gradient_checkpointing_enable()
    print("  ✅ 모델 로드 완료")
    
    # LoRA 설정
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # ══════════════════════════════════════════════
    # 4. TrainingArguments
    # ══════════════════════════════════════════════
    print("\n[4/5] 학습 설정...")
    
    training_args = TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        
        optim="adamw_torch_fused",  # H100 최적화
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        
        bf16=True,
        tf32=True,  # H100 최적화
        
        logging_steps=50,
        logging_first_step=True,
        save_strategy="epoch",
        eval_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=False,
        
        dataloader_num_workers=4,  # H100은 멀티워커 가능
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    
    # ══════════════════════════════════════════════
    # 5. 학습
    # ══════════════════════════════════════════════
    print("\n[5/5] 학습 시작...")
    print(f"\n  ⏱️ 예상 시간: 약 20~30시간 (5 Epoch)")
    print("-" * 70)
    
    epoch_callback = EpochCallback(processor, test_df)
    
    trainer = VQATrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        callbacks=[epoch_callback],
    )
    
    trainer.train()
    
    # 최종 저장
    final_path = ADAPTER_DIR / "final"
    final_path.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(str(final_path))
    processor.save_pretrained(str(final_path))
    
    print("\n" + "=" * 70)
    print("  ✅ 학습 완료!")
    print(f"\n  📂 어댑터: {ADAPTER_DIR}")
    
    csv_files = sorted(SUBMISSION_DIR.glob("submission_*.csv"))
    print(f"  📄 CSV ({len(csv_files)}개):")
    for f in csv_files:
        print(f"     · {f.name}")
    
    print("=" * 70)
    
    del model, trainer
    cleanup()


# 실행
train()
